In [1]:
# Create a Spark session
from pyspark.sql import SparkSession
spark = (
    SparkSession
    .builder
    .appName("Window Operation and Watermarks")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[2]")
    .getOrCreate()
)


spark

In [5]:
kafka_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "ed-kafka:29092")
    .option("subscribe", "wildlife")
    .option("startingOffsets", "earliest")
    .load()
)
kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [7]:
# Convert binary to string value column

from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))
kafka_json_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [12]:
from pyspark.sql.functions import from_json, col

# JSON schema
json_schema = "event_time string, data string"


# Expldoe JSON FORM Value column using schema
json_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema))
json_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)
 |-- values_json: struct (nullable = true)
 |    |-- event_time: string (nullable = true)
 |    |-- data: string (nullable = true)



In [15]:

flattnend_df = json_df.select("values_json.event_time", "values_json.data")
flattnend_df.printSchema()

root
 |-- event_time: string (nullable = true)
 |-- data: string (nullable = true)



In [18]:
# Split the data in words
from pyspark.sql.functions import col, split, explode
words_df = flattnend_df \
    .withColumn("words", split("data", " ")) \
    .withColumn("word", explode("words")) \
    .withColumn("event_time", col("event_time").cast("timestamp"))

words_df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- data: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- word: string (nullable = false)



In [19]:
# Aggregate the words to generate count

from pyspark.sql.functions import count, lit, window

df_agg = words_df \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(window("event_time", "10 minutes", "5 minutes"), "word").agg(count(lit(1)).alias("cnt"))

In [21]:
df_final = df_agg.selectExpr("window.start as start_time", "window.end as end_time", "word", "cnt")

In [22]:
df_final.printSchema()

root
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- word: string (nullable = false)
 |-- cnt: long (nullable = false)



In [25]:
{
    df_final
    .writeStream
    .format("console")
    .outputMode("complete")
    .trigger(processingTime="30 seconds")
    .option("checkpointLocation", "checkpoint_dir_kafka_2")
    .start()
}

{<pyspark.sql.streaming.StreamingQuery at 0x7fffe2345f90>}

In [26]:
{
    df_final
    .writeStream
    .format("console")
    .outputMode("update")
    .trigger(processingTime="30 seconds")
    .option("checkpointLocation", "checkpoint_dir_kafka_3")
    .start()
}

{<pyspark.sql.streaming.StreamingQuery at 0x7fffe2345270>}